In [2]:
# Define the specified date
# format yyyy-mm-dd
end_date = datetime.today() - pd.Timedelta(days = 1)
start_date = end_date - pd.Timedelta(days = 6)

print(f"Repor Start Date:",start_date)
print(f"Report End Date:",end_date)

base_path = './dengue_daily_reports_karnataka/'
new_folder_name = f'karnataka-dengue-hotspots-{start_date.strftime("%b %d")}-{end_date.strftime("%b %d")}'
save_dir = os.path.join(base_path, new_folder_name)
os.makedirs(save_dir, exist_ok=True)

data_file_path = f'{base_path}ka-ihip-ll.csv'

region_code_name_file_path = './karnataka_shape_files_taluk_district/karnataka_villages_lgd_codes.csv'

path_shape_file_vilg = './karnataka_shape_files_taluk_district/Village/Village.shp'

path_dist_shape_files = './karnataka_shape_files_taluk_district/District/District.shp'

path_taluk_shape_files = './karnataka_shape_files_taluk_district/Taluk/Taluk.shp'


Repor Start Date: 2024-09-09 10:51:56.681923
Report End Date: 2024-09-15 10:51:56.681923


In [3]:
# SHAPE FILES FOR VILLAGE TALUK AND DISTRICT

df_kar_dist_tluk_vilg_name_code = pd.read_csv(region_code_name_file_path)

df_kar_dist_name_code = df_kar_dist_tluk_vilg_name_code[['district_code', 'district']].drop_duplicates()
df_kar_dist_name_code = df_kar_dist_name_code.reset_index(drop = True)

df_kar_tluk_name_code = df_kar_dist_tluk_vilg_name_code[['district_code', 'district','taluk_code', 'taluk']].drop_duplicates()
df_kar_tluk_name_code = df_kar_tluk_name_code.reset_index(drop = True)

gdf_kar_vilg = gpd.read_file(path_shape_file_vilg)
gdf_kar_vilg_wgs84 = gdf_kar_vilg.to_crs(epsg=4326)
gdf_kar_vilg_wgs84['centroid'] = gdf_kar_vilg_wgs84['geometry'].centroid
gdf_kar_vilg_wgs84['centroid_longitude'] = gdf_kar_vilg_wgs84['centroid'].x
gdf_kar_vilg_wgs84['centroid_latitude'] = gdf_kar_vilg_wgs84['centroid'].y

gdf_map_kar_vilg_wgs84 = pd.merge(gdf_kar_vilg_wgs84,df_kar_dist_tluk_vilg_name_code, 
                              left_on='LGD_Villag',right_on='village_code',how='left')

gdf_map_kar_vilg_wgs84[['district_code', 'taluk_code', 'village_code']] = \
                        gdf_map_kar_vilg_wgs84[['district_code', 'taluk_code', 'village_code']]\
                        .fillna(-1).astype(int)

gdf_map_kar_vilg_wgs84 = gdf_map_kar_vilg_wgs84.reset_index(drop='True')

gdf_kar_dist = gpd.read_file(path_dist_shape_files)
gdf_kar_dist_wgs84 = gdf_kar_dist.to_crs(epsg=4326)

gdf_kar_tluk = gpd.read_file(path_taluk_shape_files)
gdf_kar_tluk_wgs84 =gdf_kar_tluk.to_crs(epsg=4326)
gdf_kar_tluk_wgs84.loc[gdf_kar_tluk_wgs84['KGISTalukN'] == 'Harohalli', 'LGD_TalukC'] = 7196

gdf_map_kar_tluk_wgs84 = pd.merge(gdf_kar_tluk_wgs84,df_kar_tluk_name_code, left_on='LGD_TalukC',
                                 right_on='taluk_code',how='left')

gdf_map_kar_tluk_wgs84[['district_code','taluk_code']]=gdf_map_kar_tluk_wgs84[['district_code','taluk_code']].fillna(-1).astype(int)


/var/folders/0y/f_1604916677sp_j7gkzq8hc0000gn/T/ipykernel_8477/2177124882.py:13: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf_kar_vilg_wgs84['centroid'] = gdf_kar_vilg_wgs84['geometry'].centroid


In [4]:
# LOAD CASE DATA AND FILTER 

df_kar_raw = pd.read_csv(data_file_path)
print(f'total entries:',len(df_kar_raw))
print('header names:', df_kar_raw.columns)

df_kar_raw = df_kar_raw.dropna(subset=['event.test.resultDate'])
df_kar_raw['event.test.resultDate'] = pd.to_datetime(df_kar_raw['event.test.resultDate'], errors='coerce')
df_kar_raw['test_date'] = df_kar_raw['event.test.resultDate'].dt.strftime('%Y-%m-%d')
df_kar_raw['test_date'] = pd.to_datetime(df_kar_raw['test_date'])
df_kar_raw['test_date'] = df_kar_raw['test_date'].dt.date

df_kar_raw['event.test.test1.result'] = df_kar_raw['event.test.test1.result']\
                                        .map({'Positive': 1, 'Unknown': 0})
df_kar_raw['event.test.test2.result'] = df_kar_raw['event.test.test2.result']\
                                        .map({'Positive': 1, 'Unknown': 0})

df_kar_raw['test_result'] = (df_kar_raw['event.test.test1.result'] \
                            + df_kar_raw['event.test.test2.result']).fillna(0).astype(int)

df_kar_raw[['region_admin2','district_code']] = df_kar_raw['location.admin2.ID']\
                                                 .str.split('_', expand=True)
df_kar_raw[['region_admin3','taluk_code']] = df_kar_raw['location.admin3.ID']\
                                               .str.split('_', expand=True)
df_kar_raw[['region_admin4','village_code']] = df_kar_raw['location.admin5.ID']\
                                                .str.split('_', expand=True)

df_kar_raw['district_code']=df_kar_raw['district_code'].astype(int)
df_kar_raw['taluk_code']=df_kar_raw['taluk_code'].astype(int)
df_kar_raw['village_code'] = df_kar_raw['village_code'].str.replace(r'\D', '', regex=True).astype(int)

select_columns = ['test_date','demographics.ageRange','demographics.gender','test_result',
                  'region_admin2','district_code','region_admin3','taluk_code','region_admin4','village_code',
                 'location.geometry.latitude.provided','location.geometry.longitude.provided']

df_kar_raw = df_kar_raw[select_columns].rename(columns={
                'demographics.ageRange':'age_range',
                'demographics.gender':'gender',
                'location.geometry.latitude.provided' : 'latitude',
                'location.geometry.longitude.provided' : 'longitude'
})



total entries: 18684
header names: Index(['metadata.primaryDate', 'metadata.patientID',
       'metadata.patientHealthID', 'metadata.patientTransactionID',
       'metadata.patientSpecimenID', 'metadata.diseaseName',
       'metadata.diseaseCode', 'demographics.ageRange', 'demographics.gender',
       'location.country.ID', 'location.country.name',
       'location.admin.hierarchy', 'location.admin1.ID',
       'location.admin1.name', 'location.admin2.ID', 'location.admin2.name',
       'location.admin3.ID', 'location.admin3.name', 'location.admin4.ID',
       'location.admin4.name', 'location.admin5.ID', 'location.admin5.name',
       'location.admin.coarseness', 'location.geometry.latitude.provided',
       'location.geometry.longitude.provided', 'event.symptomOnset',
       'event.symptomOnsetDate', 'event.test',
       'event.test.sampleCollectionDate', 'event.test.testingLab',
       'event.test.test1.code', 'event.test.test1.name',
       'event.test.test1.result', 'event.test.te

In [5]:
df_merge_kar_raw = pd.merge (df_kar_raw,df_kar_dist_tluk_vilg_name_code,
                          left_on=['district_code','taluk_code','village_code'],
                          right_on = ['district_code', 'taluk_code', 'village_code'],
                          how = 'left')
                     
print(df_kar_raw.columns)
print(df_kar_dist_tluk_vilg_name_code.columns)

Index(['test_date', 'age_range', 'gender', 'test_result', 'region_admin2',
       'district_code', 'region_admin3', 'taluk_code', 'region_admin4',
       'village_code', 'latitude', 'longitude'],
      dtype='object')
Index(['district_code', 'district', 'taluk_code', 'taluk', 'village_code',
       'village', 'block_panchayat_code', 'block_panchayat',
       'gram_panchayat_code', 'gram_panchayat'],
      dtype='object')


In [6]:
df_merge_kar_raw

,test_date,age_range,gender,test_result,region_admin2,district_code,region_admin3,taluk_code,region_admin4,village_code,latitude,longitude,district,taluk,village,block_panchayat_code,block_panchayat,gram_panchayat_code,gram_panchayat
0,2024-07-16,"(65, 105]",Female,1,district,528,ulb,251927,admin,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-07-17,"(45, 65]",Female,1,district,528,ulb,251927,admin,0,15.157263,76.917004,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024-07-19,"(18, 25]",Female,1,district,528,subdistrict,5501,village,938318,15.071137,76.534616,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2024-07-18,"(18, 25]",Male,1,district,528,subdistrict,5501,village,605016,NaN,NaN,Ballari,Sandur,Toranagal,6117.0,Sandur,216264.0,Torangal
4,2024-07-18,"(12, 18]",Female,1,district,528,subdistrict,7109,village,604879,15.341190,76.914928,Ballari,Kurugodu,Byluru,296803.0,Kurugodu,216135.0,Sindhigeri
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18680,2024-09-10,"(25, 45]",Male,1,district,540,subdistrict,5494,village,604386,NaN,NaN,Haveri,Hirekerur,Balambid,6188.0,Hirekerur,218556.0,Buradikatti
18681,2024-09-10,"(6, 12]",Female,1,district,540,ulb,251912,admin,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18682,2024-09-10,"(25, 45]",Male,1,district,540,subdistrict,5494,village,604368,14.516167,75.403698,Haveri,Hirekerur,Abalur,6188.0,Hirekerur,218550.0,Abalur
18683,2024-09-10,"(25, 45]",Male,1,district,540,subdistrict,5494,village,604418,14.431724,75.389424,Haveri,Hirekerur,Channalli,6188.0,Hirekerur,218557.0,Channalli


In [7]:
# SELECT DATE FOR THE ANALYSIS

start_date = pd.to_datetime(start_date).date()
end_date = pd.to_datetime(end_date).date()

df_kar_bbmp_7d = df_merge_kar_raw [(df_merge_kar_raw ['test_date'] >= start_date) &
                    (df_merge_kar_raw ['test_date'] <= end_date)]

df_kar_7d = df_kar_bbmp_7d[df_kar_bbmp_7d['district_code']!=525]
df_bbmp_7d = df_kar_bbmp_7d[df_kar_bbmp_7d['district_code']==525]

print(f"Entries with State Info",len(df_kar_7d))

# print(df_kar_7d.info())
columns_to_check = ['test_date']
df_kar_7d = df_kar_7d.dropna(subset=columns_to_check)
df_kar_7d  = df_kar_7d.reset_index(drop=True)
print(f"Entries without Test Date",len(df_kar_7d))

######################## Village wise total cases #################################

df_kar_7d_vilg = df_kar_bbmp_7d[(df_kar_bbmp_7d['village'].notna())].reset_index(drop=True)
print(f"Entries with Village Name Info",len(df_kar_7d_vilg))

df_kar_7d_vilg_nan = df_kar_bbmp_7d[(df_kar_bbmp_7d['village'].isna())].reset_index(drop=True)
print(f"Entries WithOut Village Info",len(df_kar_7d_vilg_nan))

df_kar_7d_latlong_info = df_kar_7d[(df_kar_7d['latitude'].notna())\
                                   & (df_kar_7d['longitude'].notna())\
                                   & (df_kar_7d['village'].isna())\
                                   & (df_kar_7d['region_admin3']!='ulb')]

print(f"Entries With Lat-Long Info",len(df_kar_7d_latlong_info))

df_kar_7d_latlong_info = df_kar_7d_latlong_info[(df_kar_7d_latlong_info['latitude'] >= -90)\
                                & (df_kar_7d_latlong_info['latitude'] <= 90)]
df_kar_7d_latlong_info = df_kar_7d_latlong_info[(df_kar_7d_latlong_info['longitude'] >= -180) \
                                & (df_kar_7d_latlong_info['longitude'] <= 180)]

print(f"Entries With Lat-Long Info",len(df_kar_7d_latlong_info))

geometry = [Point(xy) for xy in zip(df_kar_7d_latlong_info['longitude'],\
                                    df_kar_7d_latlong_info['latitude'])]

df_kar_7d_latlong_info = gpd.GeoDataFrame(df_kar_7d_latlong_info,crs='EPSG:4326', 
                                               geometry=geometry)

gdf_map_latlong_vilg = gpd.sjoin(df_kar_7d_latlong_info, gdf_map_kar_vilg_wgs84,
                                   how='left', predicate='within')

gdf_map_latlong_vilg = gdf_map_latlong_vilg.dropna(subset=['village_right'])

selected_column = ['district_code_left','taluk_code_left','village_code_left',
                   'LGD_Villag','district_code_right','taluk_code_right','village_code_right']

gdf_map_latlong_vilg[selected_column] = gdf_map_latlong_vilg[selected_column].fillna(0).astype(int)

selected_columns = ['test_date',  'age_range', 'gender', 'test_result',
       'region_admin2', 'region_admin3', 'region_admin4', 'latitude',
       'longitude', 'district_code_right',
       'district_right', 'taluk_code_right', 'taluk_right',
       'village_code_right', 'village_right']


gdf_map_latlong_vilg = gdf_map_latlong_vilg[selected_columns]\
                     .rename(columns={'test_date':'test_date','age_range':'age_range',
                                      'gender':'gender','test_result':'test_result',
                                      'region_admin2':'region_admin2',
                                      'district_code_right':'district_code',
                                      'region_admin3':'region_admin3',
                                      'taluk_code_right':'taluk_code',
                                      'region_admin4':'region_admin4',
                                      'village_code_right':'village_code',
                                      'latitude':'latitude', 
                                      'longitude':'longitude',
                                      'district_right':'district',
                                      'taluk_right':'taluk',
                                      'village_right':'village'
                                      })

# # Concatenate the two DataFrames
df_kar_vilg_all = pd.concat([df_kar_7d_vilg, gdf_map_latlong_vilg])
df_kar_vilg_all = df_kar_vilg_all.reset_index(drop = True)
len(df_kar_vilg_all)
df_kar_vilg_all

df_kar_vilg_cases = df_kar_vilg_all.groupby(['district_code','district',
                                                  'taluk_code','taluk','village_code','village'])\
                                                .size().reset_index(name='cases')

df_kar_vilg_cases['label'] = df_kar_vilg_cases['cases']\
                                    .apply(lambda x: 'hotspot' if x > 1 else 'non-hotspot')

df_kar_vilg_cases['hotspot'] = (df_kar_vilg_cases['cases'] > 1).astype(int)
df_kar_vilg_cases['non_hotspot'] = (df_kar_vilg_cases['cases'] <= 1).astype(int)

df_kar_vilg_hotspot = df_kar_vilg_cases.query('hotspot == 1').reset_index(drop=True)


######################## State total cases #################################

df_kar_state_total_cases = df_kar_vilg_cases['cases'].sum()

######################## District wise total cases #################################
df_kar_dist_total_cases = df_kar_vilg_cases.groupby(['district_code', 'district'])['cases'].sum().reset_index()
# df_kar_dist_total_cases = pd.merge(df_kar_dist_name_code,kar_dist_total_cases,
#                                on=['district_code', 'district'], how='left')
df_kar_dist_total_cases['test_result'] = df_kar_dist_total_cases['cases'].fillna(0).astype(int)

df_kar_dist_total_cases = df_kar_dist_total_cases.rename(columns={
    'district_code': 'District Code',
    'district': 'District',
    'test_result': 'Cases',
})

df_kar_dist_total_cases = df_kar_vilg_cases.groupby(['district_code', 'district'])['cases'].sum().reset_index()
df_kar_dist_total_hotspot = df_kar_vilg_cases.groupby(['district_code', 'district'])['hotspot'].sum().reset_index()
df_kar_dist_total_case_hotspot = pd.merge(df_kar_dist_total_cases,df_kar_dist_total_hotspot,
                   on = ['district_code','district'],
                   how = 'inner')

######################## Taluk wise total cases #################################

df_kar_tluk_total_cases = df_kar_vilg_cases.groupby(['district_code', 'district','taluk_code','taluk'])\
                                    ['cases'].sum().reset_index()
df_kar_tluk_total_cases['cases'] = df_kar_tluk_total_cases['cases'].fillna(0).astype(int)

# df_kar_tluk_total_cases = pd.merge(df_kar_tluk_name_code,kar_tluk_total_cases,
#                                on=['district_code', 'district','taluk_code','taluk'], how='left')
# df_kar_tluk_total_cases['test_result'] = df_kar_tluk_total_cases['test_result'].fillna(0).astype(int)

df_kar_tluk_total_cases = df_kar_tluk_total_cases.rename(columns={
    'district_code':'District Code',
    'district': 'District',
    'taluk_code': 'Taluk Code',
    'taluk': 'Taluk',
    'cases':'Cases',
})
df_kar_tluk_total_cases

Entries with State Info 485
Entries without Test Date 485
Entries with Village Name Info 229
Entries WithOut Village Info 835
Entries With Lat-Long Info 45
Entries With Lat-Long Info 45


,District Code,District,Taluk Code,Taluk,Cases
0,524,Bagalkote,5444,Bilgi,1
1,524,Bagalkote,5445,Mudhol,1
2,524,Bagalkote,5446,Badami,2
3,524,Bagalkote,5447,Bagalkot,6
4,524,Bagalkote,5448,Hungund,1
...,...,...,...,...,...
102,631,Ramanagara,5608,Kanakapura,1
103,635,Yadgir,5587,Shorapur,1
104,635,Yadgir,5588,Shahpur,2
105,635,Yadgir,5589,Yadgir,1


In [8]:
# HOTSPOT CALCULATION FOR THE DAY

# this_date  = pd.to_datetime(end_date)
df_kar_enddate  = df_kar_vilg_all[df_kar_vilg_all['test_date']==end_date]

df_kar_enddate = df_kar_enddate.reset_index(drop =True)
df_kar_enddate_cases = df_kar_enddate.groupby(['district_code','district','taluk_code','taluk','village_code','village']).size().reset_index(name='cases')

df_kar_dist_enddate_cases = df_kar_enddate.groupby(['district_code','district'])['test_result'].sum().reset_index()

df_kar_enddate_cases['hotspot'] = (df_kar_enddate_cases['cases'] > 1).astype(int)
df_kar_enddate_hotspot = df_kar_enddate_cases.query('hotspot == 1').reset_index(drop=True)

df_kar_dist_enddate_hotspot = df_kar_enddate_hotspot.groupby(['district_code','district'])['hotspot'].sum().reset_index()

df_all_dist_enddate_cases =  pd.merge(df_kar_dist_name_code,df_kar_dist_enddate_cases,
                                  on = ['district_code','district'],how ='left')

df_all_dist_enddate_case_hotspot = pd.merge(df_all_dist_enddate_cases, df_kar_dist_enddate_hotspot,
                           on = ['district_code','district'], how= 'left')

df_all_dist_enddate_case_hotspot[['test_result','hotspot']] = df_all_dist_enddate_case_hotspot[['test_result','hotspot']]\
                                                           .fillna(0).astype(int)

######## drop BBMP Urban #####################################################################

# df_all_dist_enddate_case_hotspot = df_all_dist_enddate_case_hotspot\
#                                     [df_all_dist_enddate_case_hotspot['district_code']!=525]
# df_all_dist_enddate_case_hotspot = df_all_dist_enddate_case_hotspot.reset_index(drop=True)
df_all_dist_enddate_case_hotspot 

,district_code,district,test_result,hotspot
0,536,Dharwad,0,0
1,635,Yadgir,0,0
2,546,Raichur,0,0
3,535,Davangere,0,0
4,532,Chikkamagaluru,0,0
5,541,Kodagu,0,0
6,548,Tumakuru,0,0
7,529,Bidar,0,0
8,543,Koppal,0,0
9,630,Chikkaballapura,0,0


In [9]:
if not df_all_dist_enddate_case_hotspot.empty:
    
    gdf_map_hotspot_wgs84 = pd.merge(df_kar_vilg_hotspot, gdf_map_kar_vilg_wgs84, left_on=['district_code','district',
                                                                  'taluk_code','taluk' ,
                                                                   'village_code','village'], 
                                                        right_on=['district_code','district',
                                                                    'taluk_code','taluk' ,
                                                                  'village_code','village'], 
                                                            how='inner')

selected_columns = ['district_code',
                    'district',
                    'taluk_code',
                    'taluk',
                    'village_code',
                    'village',
                    'cases']

hotspot_table = gdf_map_hotspot_wgs84[selected_columns].rename(columns={
    'district':'District',
    'taluk': 'Taluk',   
    'village_code': 'Village LGD Code',
    'village': 'Hotspot Villages',
    'cases': 'Cases',
})

In [10]:
# PLOT HOTSPOT AND TALUK LEVEL CASES 

from pandas.plotting import table
from matplotlib.gridspec import GridSpec
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
import numpy as np

from matplotlib import gridspec


district_codes = hotspot_table['district_code'].unique()
# district_codes = [534]

for district_code in district_codes:



    gdf_dist_wgs84_filt =  gdf_kar_dist_wgs84[gdf_kar_dist_wgs84['LGD_Distri']== district_code]
    gdf_taluk_wgs84_filt = gdf_map_kar_tluk_wgs84[gdf_map_kar_tluk_wgs84['district_code']== district_code]
    gdf_vilg_wgs84_filt = gdf_map_kar_vilg_wgs84[gdf_map_kar_vilg_wgs84['district_code']==district_code]
   
    district_name = gdf_dist_wgs84_filt['KGISDist_1'].iloc[0] 
    x_min, y_min, x_max, y_max = gdf_dist_wgs84_filt.total_bounds
    

    df_kar_tluk_total_cases_filt = df_kar_tluk_total_cases[df_kar_tluk_total_cases['District Code'] == district_code]
    # gdf_map_kar_tluk_total_cases_filt = pd.merge(df_kar_tluk_total_cases_filt, gdf_taluk_wgs84_filt, left_on='taluk_code', 
    #                        right_on='LGD_TalukC', how='inner')

    hotspot_table_dist = hotspot_table[hotspot_table['district_code'] == district_code]
    hotspot_table_dist = hotspot_table_dist.reset_index(drop=True)
    hotspot_table_dist['Cases']= hotspot_table_dist['Cases'].astype(int)
    
    gdf_hotspots_dist = gdf_map_hotspot_wgs84[gdf_map_hotspot_wgs84['district_code'] == district_code]
    gdf_hotspots_dist = gdf_hotspots_dist.reset_index(drop=True)
    gdf_hotspots_dist = gdf_hotspots_dist.drop_duplicates(subset=['village', 'village_code'],
                                                           keep='first').reset_index(drop=True)
    

    gdf_hotspots_dist['centroid_longitude'] = gdf_hotspots_dist['centroid_longitude'].astype(float)
    gdf_hotspots_dist['centroid_latitude'] = gdf_hotspots_dist['centroid_latitude'].astype(float)
   

    fig = plt.figure(figsize=(16, 9))
    gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])

    ax1 = fig.add_subplot(gs[0])
    ax2 = fig.add_subplot(gs[1])


############################# COLOR MAP DENGUE CASES AND HOTSPOT MAPPING ################

    
    gdf_vilg_wgs84_filt.plot(ax=ax1, color='white', edgecolor=(0.4, 0.4, 0.4, 0.5), 
                        alpha=0.5, linewidth=0.6)  

    gdf_taluk_wgs84_filt.plot(ax=ax1, color='white', edgecolor=(0.0, 0.8, 0.8, 0.5), 
                        alpha=0.5, linewidth=0.8)
    
    colors = [
        (255/255, 245/255, 235/255, 0.8),  
        (253/255, 208/255, 162/255, 0.8),   
        (253/255, 174/255, 107/255, 0.8), 
        (253/255, 141/255, 60/255, 0.8),
        (241/255,105/255,19/255,0.8)
    ] 
    cases_min = df_kar_tluk_total_cases_filt['Cases'].min()
    cases_max = df_kar_tluk_total_cases_filt['Cases'].max()

    if cases_max < 5:
        cases_min = 0
        cases_max = 5
        ratio = (cases_max - cases_min) / 5
        norm = Normalize(vmin=cases_min, vmax=cases_max)
        cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=5)
        ticks = [0, 1, 2, 3, 4, 5]
        tick_labels = [str(t) for t in ticks]
        
    else:
        ratio = (cases_max - cases_min) / 5
        norm = Normalize(vmin=cases_min, vmax=cases_max)
        cmap = LinearSegmentedColormap.from_list('custom_cmap', colors, N=5)
        ticks = [cases_min + (i * (cases_max - cases_min) / 5) for i in range(6)]
        tick_labels = [f'{int(t)}' for t in ticks]
    
    for idx, row in df_kar_tluk_total_cases_filt.iterrows():
        cases = row['Cases']
        color = cmap(norm(cases))
        gdf_taluk_wgs84_filt.loc[gdf_taluk_wgs84_filt['taluk_code'] == row['Taluk Code']]\
                            .plot(ax= ax1, color=color, edgecolor=(0.0, 0.8, 0.8),alpha=0.5)
        
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax= ax1, orientation='vertical', shrink=0.8)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels(tick_labels)
    cbar.ax.tick_params(labelsize=16)

    pos = ax1.get_position()
    left, bottom, width, height = pos.x0, pos.y0, pos.width, pos.height
    cbar.ax.set_position([left + width + 0.02, bottom, 0.02, height])  # [left, bottom, width, height]
 #############################
    for idx, row in gdf_taluk_wgs84_filt.iterrows():
        
        taluk_name = row['KGISTalukN']  
        x, y = row.geometry.centroid.x, row.geometry.centroid.y  
        ax1.plot(x, y, marker='*', markersize=3, markerfacecolor='none', markeredgecolor=(0, 0, 0, 0.5))

        ax1.annotate(taluk_name, xy=(x, y), xytext=(3, 3), 
                textcoords='offset points', fontsize=8, 
                color=(0, 0, 0, 0.8),ha='center')
    
    gdf_dist_wgs84_filt.plot(ax=ax1, color='white', edgecolor=(0.3, 0.3, 0.8, 0.7), 
                        alpha=0.0, linewidth=1)  # Adjust linewidth as needed
    
    # unique_villages_b = table_df_dist.drop_duplicates(subset=['Hotspot Villages', 'Village LGD Code'], keep='first').reset_index(drop=True)

    bubble_size = gdf_hotspots_dist['cases'] * 20
    ax1.scatter(gdf_hotspots_dist['centroid_longitude'],  gdf_hotspots_dist['centroid_latitude'], color=(1, 0.2, 0.2), 
           s=bubble_size, alpha=0.8,label='Hotspots')


    # ax1.set_title(textstr, fontsize=12, loc='left')
    ax1.set_xlim(x_min, x_max)
    ax1.set_ylim(y_min, y_max)
    ax1.set_xticklabels([])
    ax1.set_yticklabels([])
    ax1.tick_params(axis='both', which='both', length=0)
    
############################# COLOR MAP DENGUE CASES AND HOTSPOT MAPPING ################
    
####################### TABLE HOTSPOTS #################################################
    if hotspot_table_dist.empty:
        ax2.text(0.5, 0.5, "This district doesn't have dengue hotspots", ha='center', va='center', fontsize=12)
        ax2.axis('off')
    else:
        district_name = hotspot_table_dist['District'].unique()[0]
        hotspot_table_dist_filt = hotspot_table_dist.drop_duplicates(subset=['Hotspot Villages', 
                                  'Village LGD Code'], keep='first').reset_index(drop=True)

    # Drop unnecessary columns
        hotspot_table_dist_filt = hotspot_table_dist_filt.drop(columns=['district_code',
                                        'Village LGD Code','District','taluk_code'])

        hotspot_table_dist_filt = hotspot_table_dist_filt.rename(columns={
                                                        'Taluk': 'Taluk',
                                                    'Hotspot Villages': 'Hotspot Villages',
                                                    'Cases': 'Cases in the \nHotspot',
                                                    })
        total_hotspot_district = len(hotspot_table_dist_filt)
    
# Hide the axes
        ax2.xaxis.set_visible(False)
        ax2.yaxis.set_visible(False)
        ax2.set_frame_on(False)

    # Create a table
        table = ax2.table(cellText=hotspot_table_dist_filt.values,
                     colLabels=hotspot_table_dist_filt.columns,
                     cellLoc='center', loc='center')

        col_widths = [0.35, 0.45, 0.28]  # Adjust these values based on your data
   
        for i, width in enumerate(col_widths):
            for j in range(len(hotspot_table_dist_filt) + 1):  # +1 for the header
                cell = table[j, i]
                cell.set_width(width)
                if j == 0:
                    cell.set_height(0.1)  # Adjust the header height as needed
                else:
                    cell.set_height(0.075)  # Adjust the row height as needed

    # Style the table
        table.auto_set_font_size(False)
        table.set_fontsize(15)
        table.scale(1, 1)  # Adjust scaling as needed

        table_edge_color = (150/255, 150/255, 150/255,0.7) 
        cell_dict = table.get_celld()
        for (i, j), cell in cell_dict.items():
            cell.set_edgecolor(table_edge_color)
            cell.set_linewidth(0.25)  # Adjust this value to change line width

    # Set Sans-Serif font family for all cells
        for key, cell in table.get_celld().items():
            cell.set_text_props(fontfamily='sans-serif')
    
       
####################### FIGURE TITLE #################################################
    total_cases_district = df_kar_dist_total_cases[df_kar_dist_total_cases['district_code'] == district_code]\
                            .iloc[0]['cases']  
    total_hotspot_district = df_kar_dist_total_hotspot[df_kar_dist_total_hotspot['district_code'] == district_code]\
                            .iloc[0]['hotspot']
          
    title_str = '\n'.join((f"Date: {start_date.strftime('%b %d')} to {end_date.strftime('%b %d')} (past 7 days)",
            f"District Name: {district_name}",
            f"Total Cases in the District: {total_cases_district}",
            f"Total Number of Hotspots: {total_hotspot_district} (two or morecases in a village)",
            ))
    
    fig.suptitle(title_str,fontsize=14,fontweight='bold', y=1)

    ax1.set_title('Taluk level total cases (past 7 days)',fontsize=12)
    # ax2.set_title('Hotspot Villages (within past 7 days)',fontsize=12)
    # plt.tight_layout(rect=[0, 0, 1, 0.95])

###################### SAVE FIGURE ##########################################################
    save_hotspot_fig = f'{save_dir}/{district_name}_hotspot_map.png'
    plt.savefig(save_hotspot_fig, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.gridspec as gridspec


# Sort both DataFrames by the specified column

selected_column = ['district', 'cases', 'hotspot']
df_table_ax1 = df_kar_dist_total_case_hotspot[selected_column]\
                                .rename(columns={'district': 'District', 'cases': 'Cases', 'hotspot': 'Hotspot'})

total_case_state = df_table_ax1['Cases'].sum()
total_hotspot_state = df_table_ax1['Hotspot'].sum()

selected_column = ['district', 'test_result', 'hotspot']
df_table_ax2 = df_all_dist_enddate_case_hotspot[selected_column]\
                                .rename(columns={'district': 'District', 'test_result': 'Cases', 'hotspot': 'Hotspot'})

df_table_ax1 = df_table_ax1.sort_values(by='District')
df_table_ax2 = df_table_ax2.sort_values(by='District')


def create_and_style_table(ax, df, col_widths, font_size=15, line_width=0.25, table_edge_color=(150/255, 150/255, 150/255, 0.7)):
    ax.xaxis.set_visible(False)
    ax.yaxis.set_visible(False)
    ax.set_frame_on(False)

    table = ax.table(cellText=df.values,
                     colLabels=df.columns,
                     cellLoc='center', loc='center')

    for i, width in enumerate(col_widths):
        for j in range(len(df) + 1):  # +1 for the header
            cell = table[j, i]
            cell.set_width(width)
            cell.set_height(0.1 if j == 0 else 0.065)  # Adjust the header and row height as needed

    table.auto_set_font_size(False)
    table.set_fontsize(font_size)
    table.scale(1, 1)  # Adjust scaling as needed

    cell_dict = table.get_celld()
    for (i, j), cell in cell_dict.items():
        cell.set_edgecolor(table_edge_color)
        cell.set_linewidth(line_width)

    for key, cell in table.get_celld().items():
        cell.set_text_props(fontfamily='sans-serif')

fig = plt.figure(figsize=(16, 9))
gs = gridspec.GridSpec(1, 2, width_ratios=[1, 1])

ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

col_widths = [0.4, 0.25, 0.25]  # Adjust these values based on your data

create_and_style_table(ax1, df_table_ax1, col_widths)
create_and_style_table(ax2, df_table_ax2, col_widths)

title_str = '\n'.join((f"Date: {start_date.strftime('%b %d')} to {end_date.strftime('%b %d')} (past 7 days)",
            f"Total Cases in the State: {total_case_state}",
            f"Total Hotspots in the State: {total_hotspot_state}",
            ))
    
fig.suptitle(title_str,fontsize=14,fontweight='bold', y=1.45)

ax1.set_title(f"District wise Cases and Hotspot between {start_date.strftime('%b %d')} to {end_date.strftime('%b %d')}",fontsize=12, y = 1.55)
ax2.set_title(f'District wise Cases and Hotspot on {end_date.strftime("%b %d")}',fontsize=12, y = 1.55)

# Adjust spacing between subplots
plt.subplots_adjust(wspace=0.05)  # Adjust the value of wspace to reduce/increase space between subplots

###################### SAVE FIGURE ##########################################################
save_hotspot_fig = f'{save_dir}/state_cases_hotspot_map.png'
plt.savefig(save_hotspot_fig, dpi=300, bbox_inches='tight')
plt.close(fig)